# 🛶 Notebook 2: Bulkheads at request-handling time

An API service calls **payments** and **recommendations**. If *recs* gets slow, the API should still be able to process payments.

## 🛠️ Setup

```bash
cd 05-microservices/bulkhead
uv sync
```

Select the `.venv` kernel in VS Code (top-right of the notebook). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


In [ ]:
import time, random
from concurrent.futures import ThreadPoolExecutor, TimeoutError

def payments():
    time.sleep(0.05); return 'paid'
def recommendations():
    time.sleep(3); return 'recs'  # got slow today

# Two *separate* pools — bulkheads.
pay_pool = ThreadPoolExecutor(max_workers=8, thread_name_prefix='pay')
rec_pool = ThreadPoolExecutor(max_workers=4, thread_name_prefix='rec')

def handle_request():
    # Fire both in parallel; each in its own bulkhead.
    f_pay = pay_pool.submit(payments)
    f_rec = rec_pool.submit(recommendations)
    paid = f_pay.result(timeout=1)
    try:
        recs = f_rec.result(timeout=0.5)  # fail fast on recs
    except TimeoutError:
        recs = 'RECS_UNAVAILABLE'
    return {'paid': paid, 'recs': recs}

for _ in range(3):
    print(handle_request())


### Design tips
- Size each pool by the **downstream's max safe concurrency**, not by wishful thinking.
- Pair with a short timeout so slow calls don't pin threads.
- A graceful fallback for the slow dep (here: `RECS_UNAVAILABLE`) keeps the main flow (payments) working.